# 01 — Data Cleaning

**Project:** Australian Retail Trade Performance Dashboard  
**Source:** ABS Retail Trade, Cat. 8501.0 (`850101.xlsx`)  
**Output:** `data/processed/retail_clean.csv` · `data/database/retail.db`

---

## What this notebook does

Transforms the raw ABS Excel file from wide format (one column per series) into a clean long-format CSV (one row per observation), then loads it into a SQLite database for SQL analysis.

| Step | Action | Output |
|---|---|---|
| 1 | Build series lookup from Index sheet | `lookup` dict |
| 2 | Read Data1 sheet | `df_data` (519 rows × 21 cols) |
| 3 | Melt wide → long format | `df_long` (10,899 rows) |
| 4 | Join lookup, filter to Original series | `df_clean` (3,633 rows) |
| 5 | Add helper date columns | year, month, month_name |
| 6 | Validate and save | `retail_clean.csv` |
| 7 | Load into SQLite | `retail.db` |

## Setup

All file paths are built relative to the project root so this notebook works on any machine after cloning.

In [1]:
import pandas as pd
import sqlite3
import os
from pathlib import Path

# Build paths relative to this notebook's location
# notebooks/ is one level inside the project root, so .parent goes up one level
PROJECT_ROOT   = Path.cwd().parent
DATA_RAW       = PROJECT_ROOT / 'data' / 'raw'
DATA_PROCESSED = PROJECT_ROOT / 'data' / 'processed'
DATA_DB        = PROJECT_ROOT / 'data' / 'database'

# Create processed/ and database/ folders if they don't exist yet
os.makedirs(DATA_PROCESSED, exist_ok=True)
os.makedirs(DATA_DB, exist_ok=True)

print('Project root:', PROJECT_ROOT)
print('Raw data:    ', DATA_RAW)
print('Processed:   ', DATA_PROCESSED)

Project root: /Users/sofiaconcepcion/Documents/GitHub/Australian-Retail-Trade-Performance
Raw data:     /Users/sofiaconcepcion/Documents/GitHub/Australian-Retail-Trade-Performance/data/raw
Processed:    /Users/sofiaconcepcion/Documents/GitHub/Australian-Retail-Trade-Performance/data/processed


In [2]:
import pandas as pd
import os
from pathlib import Path
PROJECT_ROOT = Path.cwd().parent
DATA_RAW = PROJECT_ROOT / 'data' / 'raw'

# Create dataframe from raw excel file
df_index = pd.read_excel(DATA_RAW / '850101.xlsx', sheet_name = 'Index')

# Keep only relevant columns and rows
meta = df_index.iloc[10:, [0, 3, 4]].copy()

# Label columns
meta.columns = ['description', 'series_type', 'series_id']

# Drop with no series_id
meta = meta.dropna(subset=['series_id'])

'''accepts description input as str to output series cateogry as str'''
def extract_category(desc):
    parts = [p.strip() for p in str(desc).split(';')]
    return parts[2].strip() if len(parts) > 2 else desc

# Create new column containing cateogry of series based on description
meta['category'] = meta['description'].apply(extract_category)

# Store look up data
lookup = meta.set_index('series_id')[['category', 'series_type']]


## Step 1 — Build series lookup from the Index sheet

The ABS file contains 21 series across 3 types: **Original**, Seasonally Adjusted, and Trend.
The `Index` sheet maps every Series ID to its category name and type — we use it as a lookup
table so we can label and filter the data columns in Data1 without parsing column headers.

We only keep **Original** series — the raw numbers as collected. Seasonally Adjusted and Trend
series have already been transformed by the ABS and would distort our own analysis.

In [3]:
# Read the Index sheet — header=None because row 0 is not a header, it's metadata
df_index = pd.read_excel(DATA_RAW / '850101.xlsx', sheet_name='Index', header=None)

# Data starts at row 10 (0-indexed). Columns we need:
# 0 = description, 3 = series_type, 4 = series_id
meta = df_index.iloc[10:, [0, 3, 4]].copy()
meta.columns = ['description', 'series_type', 'series_id']

# Drop the last row which is an ABS copyright notice, not a series
meta = meta.dropna(subset=['series_id'])

def extract_category(desc):
    """Parse category name from ABS description string.
    
    ABS descriptions follow the format:
    'Turnover ;  Total (State) ;  Food retailing ;'
    We split on ';' and return the third segment (index 2).
    """
    parts = [p.strip() for p in str(desc).split(';')]
    return parts[2].strip() if len(parts) > 2 else desc

meta['category'] = meta['description'].apply(extract_category)

# Build lookup: series_id → (category, series_type)
lookup = meta.set_index('series_id')[['category', 'series_type']]

print(f'Total series: {len(lookup)}')
print(lookup['series_type'].value_counts())
print('\nSeries lookup:')
print(lookup)

Total series: 21
series_type
Original               7
Seasonally Adjusted    7
Trend                  7
Name: count, dtype: int64

Series lookup:
                                                    category  \
series_id                                                      
A3348591K                                     Food retailing   
A3348600A                          Household goods retailing   
A3348609W  Clothing, footwear and personal accessory reta...   
A3348618X                                  Department stores   
A3348627A                                    Other retailing   
A3348636C      Cafes, restaurants and takeaway food services   
A3348582J                                   Total (Industry)   
A3348594T                                     Food retailing   
A3348603J                          Household goods retailing   
A3348612K  Clothing, footwear and personal accessory reta...   
A3348621L                                  Department stores   
A3348630R             

## Step 2 — Read Data1

Data1 is in **wide format** — 519 rows (one per month, Apr 1982 → Jun 2025) and 21 columns
(one per series ID). Row 9 (0-indexed) contains the Series IDs which become our column headers.
Dates in column 0 are already Python datetime objects — no string parsing needed.

In [4]:
df_data = pd.read_excel(
    DATA_RAW / '850101.xlsx',
    sheet_name='Data1',
    header=9,       # row 9 = Series IDs → column names
    index_col=0     # column 0 = dates → row index
)

# Ensure index is parsed as datetime and drop any non-date rows
df_data.index = pd.to_datetime(df_data.index, errors='coerce')
df_data = df_data[df_data.index.notna()]
df_data.index.name = 'date'

print(f'Shape: {df_data.shape}')          # expected: (519, 21)
print(f'Date range: {df_data.index.min().strftime("%b %Y")} → {df_data.index.max().strftime("%b %Y")}')
df_data.head()

Shape: (519, 21)
Date range: Apr 1982 → Jun 2025


,A3348591K,A3348600A,A3348609W,A3348618X,A3348627A,A3348636C,A3348582J,A3348594T,A3348603J,A3348612K,...,A3348630R,A3348639K,A3348585R,A3348597X,A3348606R,A3348615T,A3348624V,A3348633W,A3348642X,A3348588W
date,,,,,,,,,,,,,,,,,,,,,
1982-04-01,1162.6,592.3,359.9,460.1,479.1,342.4,3396.4,1166.8,653.4,360.6,...,507.8,349.7,3518.7,1172.8,652.8,362.6,483.1,505.0,347.5,3523.4
1982-05-01,1150.9,629.6,386.6,502.6,486.1,342.1,3497.9,1178.2,648.7,362.5,...,502.2,346.3,3527.6,1181.3,654.1,361.9,484.8,504.8,346.3,3533.6
1982-06-01,1160.0,607.4,350.5,443.8,467.5,328.7,3357.8,1203.4,655.7,365.1,...,506.8,350.8,3561.5,1192.4,655.7,361.8,487.0,504.6,345.8,3547.0
1982-07-01,1206.4,632.4,359.3,459.1,491.1,338.5,3486.8,1209.5,660.5,361.9,...,503.6,341.5,3553.9,1202.8,656.6,361.5,489.1,505.3,345.4,3560.6
1982-08-01,1152.5,622.6,325.2,438.4,485.7,331.5,3355.9,1198.2,659.8,359.2,...,505.9,342.6,3581.8,1213.2,656.5,361.9,490.3,505.5,346.3,3573.6


## Step 3 — Melt wide → long format

Wide format (one column per series) is impossible to filter, group, or visualise dynamically.
We reshape to **long format** — one row per observation — using `melt()`.

Before: 519 rows × 21 columns = 10,899 values spread across columns  
After: 10,899 rows × 3 columns — every number has its own row

In [5]:
# reset_index() turns the date index into a regular column so melt() can see it
df_long = df_data.reset_index().melt(
    id_vars=['date'],       # keep date as-is
    var_name='series_id',   # old column names → new 'series_id' column
    value_name='turnover_m' # values → new 'turnover_m' column
)

print(f'Shape after melt: {df_long.shape}')  # expected: (10,899, 3)
df_long.head()

Shape after melt: (10899, 3)


,date,series_id,turnover_m
0,1982-04-01,A3348591K,1162.6
1,1982-05-01,A3348591K,1150.9
2,1982-06-01,A3348591K,1160.0
3,1982-07-01,A3348591K,1206.4
4,1982-08-01,A3348591K,1152.5


## Step 4 — Join lookup and filter to Original series only

Join the lookup table to add `category` and `series_type` labels to every row,
then filter to Original series only — dropping Seasonally Adjusted and Trend duplicates.

10,899 rows → 3,633 rows (exactly one-third, because we had 3 series types).

In [6]:
# Join on series_id to add category and series_type to every row
df_long = df_long.join(lookup, on='series_id')

# Keep Original series only
# .copy() avoids a pandas SettingWithCopyWarning on subsequent assignments
df_clean = df_long[df_long['series_type'] == 'Original'].copy()

print(f'Rows after filtering to Original: {len(df_clean)}')
print(f'Categories: {sorted(df_clean["category"].unique())}')

Rows after filtering to Original: 3633
Categories: ['Cafes, restaurants and takeaway food services', 'Clothing, footwear and personal accessory retailing', 'Department stores', 'Food retailing', 'Household goods retailing', 'Other retailing', 'Total (Industry)']


## Step 5 — Add helper date columns

Derived date columns make filtering and grouping easier in SQL queries and Power BI slicers
without requiring date functions at query time.

In [7]:
df_clean['year']       = df_clean['date'].dt.year        # e.g. 2024
df_clean['month']      = df_clean['date'].dt.month       # e.g. 6
df_clean['month_name'] = df_clean['date'].dt.strftime('%b')  # e.g. 'Jun'

df_clean.head()

,date,series_id,turnover_m,category,series_type,year,month,month_name
0,1982-04-01,A3348591K,1162.6,Food retailing,Original,1982,4,Apr
1,1982-05-01,A3348591K,1150.9,Food retailing,Original,1982,5,May
2,1982-06-01,A3348591K,1160.0,Food retailing,Original,1982,6,Jun
3,1982-07-01,A3348591K,1206.4,Food retailing,Original,1982,7,Jul
4,1982-08-01,A3348591K,1152.5,Food retailing,Original,1982,8,Aug


## Step 6 — Validate and save

Select only the 6 output columns, sort chronologically within each category,
and validate the output before saving. The `assert` statements crash loudly
if anything is wrong — much safer than silently saving bad data.

In [8]:
# Select final columns — drops series_id and series_type which are no longer needed
df_clean = df_clean[['date', 'year', 'month', 'month_name', 'category', 'turnover_m']]
df_clean = df_clean.dropna(subset=['turnover_m'])
df_clean = df_clean.sort_values(['category', 'date']).reset_index(drop=True)

# Validate before saving
assert df_clean.shape == (3633, 6),              f'Unexpected shape: {df_clean.shape}'
assert df_clean['turnover_m'].isna().sum() == 0,  'Nulls found in turnover_m'
assert df_clean['category'].nunique() == 7,       'Expected 7 categories'

df_clean.to_csv(DATA_PROCESSED / 'retail_clean.csv', index=False)

print('Saved: retail_clean.csv')
print(f'Shape: {df_clean.shape}')
print(f'Columns: {df_clean.columns.tolist()}')
print(f'Date range: {df_clean["date"].min().strftime("%b %Y")} → {df_clean["date"].max().strftime("%b %Y")}')
df_clean.head()

Saved: retail_clean.csv
Shape: (3633, 6)
Columns: ['date', 'year', 'month', 'month_name', 'category', 'turnover_m']
Date range: Apr 1982 → Jun 2025


,date,year,month,month_name,category,turnover_m
0,1982-04-01,1982,4,Apr,"Cafes, restaurants and takeaway food services",342.4
1,1982-05-01,1982,5,May,"Cafes, restaurants and takeaway food services",342.1
2,1982-06-01,1982,6,Jun,"Cafes, restaurants and takeaway food services",328.7
3,1982-07-01,1982,7,Jul,"Cafes, restaurants and takeaway food services",338.5
4,1982-08-01,1982,8,Aug,"Cafes, restaurants and takeaway food services",331.5


## Step 7 — Load into SQLite

Load `retail_clean.csv` into a SQLite database for use in the SQL analysis notebook.
SQLite is file-based and requires no server — the database is a single `.db` file
in `data/database/`. The `with` block ensures the connection closes automatically.

In [9]:
df = pd.read_csv(DATA_PROCESSED / 'retail_clean.csv')
df['date'] = pd.to_datetime(df['date'])  # re-parse dates (CSV stores them as strings)

with sqlite3.connect(DATA_DB / 'retail.db') as conn:
    # if_exists='replace' drops and recreates the table on each run
    df.to_sql('retail_turnover', conn, if_exists='replace', index=False)

    # Verify row count
    result = pd.read_sql('SELECT COUNT(*) AS n FROM retail_turnover', conn)
    print(result)  # expected: 3633

    # Verify all categories loaded
    cats = pd.read_sql('SELECT DISTINCT category FROM retail_turnover ORDER BY category', conn)
    print(cats)

print(f'\nDatabase saved to: data/database/retail.db')

      n
0  3633
                                            category
0      Cafes, restaurants and takeaway food services
1  Clothing, footwear and personal accessory reta...
2                                  Department stores
3                                     Food retailing
4                          Household goods retailing
5                                    Other retailing
6                                   Total (Industry)

Database saved to: data/database/retail.db
